In [10]:
dataset_ver = 5

In [12]:
dataset_dir = f"../../datasets/{dataset_ver}"

In [13]:
# from google.colab import drive
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import random
import io
import contextlib
import copy
import shutil
import re
from torch.utils.data import Subset
import sys
import time
import plotly.graph_objects as go

In [17]:
def visualize_flag_vertices(flag, use_topology=False):
    
    path = os.path.join(dataset_dir, "flags")
    # 1. Load Data
    flag_path = os.path.join(path, flag)
    data = np.load(flag_path)
    
    topology_path = os.path.join(dataset_dir, "topology", "topology_edge_index.npy")
    edge_index = np.load(topology_path)
    
    # 2. Extract Vertex Positions
    x = data[:, 0]
    y = data[:, 1]
    z = data[:, 2]

    # 3. Process Edges for Visualization (The "NaN" trick)
    u, v = edge_index[0], edge_index[1]
    
    x_lines = np.column_stack((x[u], x[v], np.full_like(u, np.nan, dtype=float))).flatten()
    y_lines = np.column_stack((y[u], y[v], np.full_like(u, np.nan, dtype=float))).flatten()
    z_lines = np.column_stack((z[u], z[v], np.full_like(u, np.nan, dtype=float))).flatten()

    # 4. Create Traces
    edge_trace = go.Scatter3d(
        x=x_lines, y=y_lines, z=z_lines,
        mode='lines',
        line=dict(color='black', width=2),
        name='Mesh Edges'
    )

    node_trace = go.Scatter3d(
        x=x, y=y, z=z,
        mode='markers',
        marker=dict(
            size=4,
            color=z,
            colorscale='Viridis',
            opacity=0.8
        ),
        name='Vertices'
    )

    # 5. Define Clean Layout
    axis_style = dict(
        showbackground=False,
        showgrid=False,
        showline=True,
        linecolor='black',
        linewidth=5,
        zeroline=True,
        zerolinewidth=5,
        zerolinecolor='red',
        title_font=dict(size=20)
    )

    fig = go.Figure(data=[edge_trace, node_trace])

    fig.update_layout(
        title="Flag Topology Visualization",
        width=1000, height=800,
        showlegend=False,
        scene=dict(
            xaxis={**axis_style, 'title': 'X Axis', 'linecolor': 'red'},
            yaxis={**axis_style, 'title': 'Y Axis', 'linecolor': 'green'},
            zaxis={**axis_style, 'title': 'Z Axis', 'linecolor': 'blue'},
            aspectmode='data',
            
            # --- CAMERA CONFIGURATION ---
            camera=dict(
                # Position of the camera (x,y,z). 
                # (0, -2.5, 0.5) looks from the "Front" (negative Y) and slightly up.
                eye=dict(x=0, y=-2.5, z=0.5), 
                
                # The point the camera looks at (Center of the flag usually)
                center=dict(x=0, y=0, z=0),
                
                # Which way is "Up" (Z-axis is up)
                up=dict(x=0, y=0, z=1)
            )
        ),
        margin=dict(r=0, b=0, l=0, t=40)
    )

    fig.show()

In [19]:
visualize_flag_vertices("flag_020_000.npy", use_topology=True)